# Task 3 - Agentic Workflows: Multi-Agent Financial Research System

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GimsaraK/CDAZZDEV-MLE-Gimsara_Elgiriyage/blob/main/task3_agentic/task3_multi_agent.ipynb)

CDAZZDEV Senior Machine Learning Engineer Technical Assessment - Gimsara Elgiriyage

| Part | What this notebook shows |
|---|---|
| 3A | One LangGraph research agent with five tools, a visible observe-and-replan trace, a validated three-section report, and recovery from injected tool failures |
| 3B | Two-agent pipeline (Data Analyst and Research Writer) |
| 3C | Short-term memory, persistent cache, and `agent_trace.jsonl` |

Agent LLM: Groq `openai/gpt-oss-120b`, then Groq `openai/gpt-oss-20b` (its own daily quota), then free OpenRouter models. The trace shows which model answered each turn. Ticker: **AAPL**. Not investment advice.

## 0. Setup

Install dependencies and load API keys from Colab Secrets or a local `.env` file. No credentials are hardcoded.

In [1]:
# AI-ASSISTED: Claude Code (claude-opus-5-5), Prompt: 'Implement the Task 3A plan (the plan approved in Entry 13)', Date: 2026-10-09 (see CITATIONS.md Entry 14)

# Colab clones the public repo and installs Task 3 requirements. Locally we only find the repo root.
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/GimsaraK/CDAZZDEV-MLE-Gimsara_Elgiriyage.git"
REPO_NAME = "CDAZZDEV-MLE-Gimsara_Elgiriyage"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo_root = Path("/content") / REPO_NAME
    if not repo_root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_root)], check=True)
    else:
        # A reused session keeps its old clone. Pull so a fix pushed to main is picked up.
        subprocess.run(["git", "-C", str(repo_root), "pull", "--ff-only"], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(repo_root / "task3_agentic" / "requirements.txt")],
        check=True,
    )
else:
    here = Path.cwd().resolve()
    repo_root = here if (here / "task3_agentic").exists() else here.parent
# Task 3 imports its own package and the Task 1 modules it reuses, both from the repo root.
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Running in Colab:", IN_COLAB)
commit = subprocess.run(["git", "-C", str(repo_root), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Repository commit:", commit or "unknown")
print("Repository root :", repo_root)

Running in Colab: False
Repository commit: e77f79d
Repository root : E:\Technical Assessments\CDAZZDEV-SMLE


In [2]:
# AI-ASSISTED: Claude Code (claude-opus-5-5), Prompt: 'Implement the Task 3A plan (the plan approved in Entry 13)', Date: 2026-10-09 (see CITATIONS.md Entry 14)

# Keys come from Colab Secrets or the repo-root .env. Only True/False is printed, never a key.
from importlib.metadata import version

from task3_agentic.src.llm import load_api_keys
from task3_agentic.src.logging_utils import configure_logging

configure_logging()
print("API keys present:", load_api_keys())
for package in ("langgraph", "langchain-core", "langchain-groq", "langchain-openai", "ddgs", "yfinance", "pydantic"):
    print(f"{package:>16}: {version(package)}")

API keys present: {'GROQ_API_KEY': True, 'OPENROUTER_API_KEY': True}
       langgraph: 1.2.14
  langchain-core: 1.6.8
  langchain-groq: 1.1.3
langchain-openai: 1.7.0
            ddgs: 9.16.0
        yfinance: 1.7.0
        pydantic: 2.13.5


## 1. Configuration and Trace Logger

Every tool call is written to `logs/agent_trace.jsonl` with its inputs, output (truncated to 200 characters), and wall-clock duration.

Every limit the agent runs under is a named constant in `src/config.py`. The trace logger appends one JSON line per tool call. A full top-to-bottom run of this notebook starts a fresh `logs/agent_trace.jsonl`, so the committed file holds exactly the runs shown below.

In [3]:
# AI-ASSISTED: Claude Code (claude-opus-5-5), Prompt: 'Implement the Task 3A plan (the plan approved in Entry 13)', Date: 2026-10-09 (see CITATIONS.md Entry 14)

import pandas as pd

from task3_agentic.src import config
from task3_agentic.src.tracing import TraceLogger

settings = {
    "Ticker": config.TICKER,
    "Agent LLM": config.GROQ_MODEL,
    "Fallback LLMs": ", ".join(config.GROQ_MODELS[1:] + config.OPENROUTER_FALLBACK_MODELS),
    "Reasoning effort (agent / report)": f"{config.AGENT_REASONING_EFFORT} / {config.REPORT_REASONING_EFFORT}",
    "Max tool rounds": config.MAX_TOOL_ROUNDS,
    "Extra rounds per report revision": config.REVISION_TOOL_ROUNDS,
    "Max report revisions": config.MAX_REPORT_REVISIONS,
    "Tool result cap sent to the LLM (chars)": config.TOOL_MESSAGE_MAX_CHARS,
    "Trace output truncation (chars)": config.TRACE_OUTPUT_MAX_CHARS,
    "Hedge horizon (calendar days)": config.HEDGE_HORIZON_CALENDAR_DAYS,
    "Volatility method": "std(daily log returns, ddof=1) x sqrt(252)",
    "Hedge vol check tolerance (pct points)": config.VOL_CHECK_TOLERANCE_PCT_POINTS,
}
display(pd.DataFrame(settings.items(), columns=["Setting", "Value"]))

# Fresh trace file for this notebook run.
config.TRACE_FILE.parent.mkdir(parents=True, exist_ok=True)
config.TRACE_FILE.write_text("", encoding="utf-8")
trace = TraceLogger(config.TRACE_FILE)
print("Trace file:", config.TRACE_FILE.relative_to(config.REPO_ROOT))

,Setting,Value
0,Ticker,AAPL
1,Agent LLM,openai/gpt-oss-120b
2,Fallback LLMs,"openai/gpt-oss-20b, nvidia/nemotron-3-super-12..."
3,Reasoning effort (agent / report),low / medium
4,Max tool rounds,8
5,Extra rounds per report revision,3
6,Max report revisions,2
7,Tool result cap sent to the LLM (chars),2500
8,Trace output truncation (chars),200
9,Hedge horizon (calendar days),90


Trace file: task3_agentic\logs\agent_trace.jsonl


In [ ]:
# AI-ASSISTED: Claude Code (claude-opus-5-5), Prompt: 'Implement the Task 3C plan', Date: 2026-10-09 (see CITATIONS.md Entry 18)

# 3C: cache and memory events go to their own log, so agent_trace.jsonl stays one line per tool call.
from task3_agentic.src.tracing import EventLogger, trace_line_count

config.EVENTS_FILE.write_text("", encoding="utf-8")
events = EventLogger(config.EVENTS_FILE)
print("Events file:", config.EVENTS_FILE.relative_to(config.REPO_ROOT))

# Part 3A - Tool-Using Research Agent

## 2. Tool Implementations

`get_price_data`, `get_news`, `calculate_volatility`, `llm_sentiment`, `web_search`.

Each tool returns a `ToolResult` envelope: `status` (`ok` / `empty` / `error`), a typed payload in `data`, and on failure an `error` plus a `hint` that names an alternative. Tools never raise. Any exception becomes an `error` envelope inside `run_traced`, which also times the call and writes the trace line.

| Tool | Payload type | Built on | Fallbacks inside the tool |
|---|---|---|---|
| `get_price_data(ticker, period)` | `PriceData` | Task 1 `load_market_data`, `add_indicators`, `build_summary` + yfinance `.info` | live -> Task 1 snapshot (read-only); bad period -> error with the valid list |
| `get_news(ticker, n)` | `NewsList` | Task 1 `get_headlines` | yfinance -> Google News RSS -> snapshot |
| `calculate_volatility(ticker, window)` | `VolatilityResult` | `src/volatility.py` | reuses prices already in the session; bad window -> error with the valid range |
| `llm_sentiment(headlines)` | `SentimentScore` | Task 1 `score_headlines` (Pydantic-validated, repair + provider fallback) | empty list -> hint to call `get_news` |
| `web_search(query)` | `SearchResults` | `ddgs` (the renamed `duckduckgo-search` package) | text -> news backend -> simplified query; retries on rate limits |

The cell below calls each tool directly, outside the agent, and prints the returned types.

In [4]:
# AI-ASSISTED: Claude Code (claude-opus-5-5), Prompt: 'Implement the Task 3A plan (the plan approved in Entry 13)', Date: 2026-10-09 (see CITATIONS.md Entry 14)

from task3_agentic.src.display import summarize_result
from task3_agentic.src.session import SessionContext
from task3_agentic.src.tools import ResearchTools

tool_ctx = SessionContext(config.TICKER, trace=trace, agent_name="tool_check")
toolkit = ResearchTools(tool_ctx)

price = toolkit.get_price_data(config.TICKER, "1y")
news = toolkit.get_news(config.TICKER, 10)
volatility = toolkit.calculate_volatility(config.TICKER, 20)
sentiment = toolkit.llm_sentiment([h.title for h in news.data.headlines] if news.ok else [])
search = toolkit.web_search("Apple AAPL analyst price target outlook")

rows = []
for result in (price, news, volatility, sentiment, search):
    rows.append({
        "tool": result.tool,
        "returns": f"{type(result).__name__}[{type(result.data).__name__}]",
        "status": result.status,
        "source": result.source,
        "summary": summarize_result(result),
    })
with pd.option_context("display.max_colwidth", 160):
    display(pd.DataFrame(rows))

groq chat completion failed (attempt 1/3): groq rate limited (HTTP 429) - retrying


groq chat completion failed (attempt 2/3): groq rate limited (HTTP 429) - retrying


01:49:42 | WARNING | task3.llm | openai/gpt-oss-120b skipped for this session after repeated 429s


,tool,returns,status,source,summary
0,get_price_data,ToolResult[PriceData],ok,live,"Apple Inc.: price 340.42, 1y return 32.4%, RSI 61.23, momentum Strong Bullish, fwd P/E 35.5067, margin 27.62% [source=live]"
1,get_news,ToolResult[NewsList],ok,live,"10 headlines from ['google_rss']; newest: 'Apple Rises 24% YTD: Should You Buy, Sell or Hold the AAPL Stock?'"
2,calculate_volatility,ToolResult[VolatilityResult],ok,session,"20d vol 16.8% (low, 17.5 pctile 1y); 90d 1-sigma move +/-8.33% -> band 313.2-370.0"
3,llm_sentiment,ToolResult[SentimentScore],ok,"groq/openai/gpt-oss-120b,groq/openai/gpt-oss-20b",score 0.0 (Neutral) from 10 headlines: 2 positive / 2 negative / 6 neutral
4,web_search,ToolResult[SearchResults],ok,ddgs,6 hits via text for 'Apple AAPL analyst price target outlook'; top: 'Apple (AAPL) Stock Forecast & Analyst Price Targets'


In [5]:
# AI-ASSISTED: Claude Code (claude-opus-5-5), Prompt: 'Implement the Task 3A plan (the plan approved in Entry 13)', Date: 2026-10-09 (see CITATIONS.md Entry 14)

# What the agent actually receives: compact JSON, capped at TOOL_MESSAGE_MAX_CHARS characters.
import json

print(f"calculate_volatility -> {len(volatility.to_llm_json())} chars")
print(json.dumps(json.loads(volatility.to_llm_json()), indent=2))
print(f"\nget_price_data fundamentals block:")
print(json.dumps(price.data.fundamentals.model_dump(exclude_none=True), indent=2) if price.ok else price.error)

calculate_volatility -> 527 chars
{
  "status": "ok",
  "tool": "calculate_volatility",
  "data": {
    "ticker": "AAPL",
    "window": 20,
    "as_of_date": "2026-10-08",
    "method": "std(daily log returns, ddof=1) x sqrt(252)",
    "annualised_vol_pct": 16.8,
    "daily_vol_pct": 1.06,
    "comparison_pct": {
      "20d": 16.8,
      "60d": 26.36,
      "252d": 24.75
    },
    "percentile_1y": 17.5,
    "regime": "low",
    "spot": 340.42,
    "horizon_calendar_days": 90,
    "horizon_trading_days": 62,
    "expected_move_pct": 8.33,
    "band_1sigma_low": 313.2,
    "band_1sigma_high": 370.0,
    "band_2sigma_low": 288.16,
    "max_drawdown_1y_pct": -13.8
  },
  "warnings": [],
  "source": "session"
}

get_price_data fundamentals block:
{
  "market_cap": 4968150990848.0,
  "trailing_pe": 39.039,
  "forward_pe": 35.5067,
  "profit_margin_pct": 27.62,
  "revenue_growth_pct": 16.4,
  "earnings_growth_pct": 28.7,
  "return_on_equity_pct": 148.75,
  "debt_to_equity_pct": 78.445,
  "cu

In [6]:
# AI-ASSISTED: Claude Code (claude-opus-5-5), Prompt: 'Implement the Task 3A plan (the plan approved in Entry 13)', Date: 2026-10-09 (see CITATIONS.md Entry 14)

# Bad inputs come back as error envelopes with a hint, never as exceptions.
for result in (
    toolkit.get_price_data(config.TICKER, "7y"),
    toolkit.calculate_volatility(config.TICKER, 1000),
    toolkit.llm_sentiment([]),
    toolkit.get_price_data("NOT A TICKER!", "1y"),
):
    print(f"{result.tool:<22} status={result.status:<6} error={result.error}")
    print(f"{'':<22} hint={result.hint}\n")

01:49:50 | WARNING | task3.tracing | Tool get_price_data failed: ValueError: 'NOT A TICKER!' is not a valid ticker symbol


get_price_data         status=error  error=Unsupported period '7y'
                       hint=Use one of: 1mo, 3mo, 6mo, 1y, 2y, 5y, 10y, ytd, max

calculate_volatility   status=error  error=window must be between 5 and 252 trading days, got 1000
                       hint=Use an integer window between 5 and 252, e.g. 20 or 60.

llm_sentiment          status=empty  error=No headlines to score
                       hint=Call get_news first and pass its headline titles as a list of strings.

get_price_data         status=error  error=ValueError: 'NOT A TICKER!' is not a valid ticker symbol
                       hint=Retry once (network errors are often transient). If it fails again, check the ticker symbol and use get_news or web_search for qualitative evidence while prices are unavailable.



## 3. Research Agent

The agent is a LangGraph `StateGraph` built by hand (`src/agent.py`):

```
START -> agent --(tool calls)--> tools -> agent             the LLM picks every call
           |
           +--(no tool calls / budget spent / LLM down)--> write_report -> check_report
                                                               |-- passes ---------------> END
                                                               |-- gaps, revisions left -> agent  (REPLAN)
                                                               +-- gaps, none left ------> END (warnings kept)
```

- **agent**: the LLM with all five tools bound (`bind_tools`). The system prompt states the goal and what each tool returns. It does **not** prescribe an order. Routing only follows the model's own tool calls.
- **tools**: LangGraph `ToolNode`. Arguments that fail the schema, or an unknown tool name, come back to the model as an error message.
- **write_report**: structured output into the Pydantic `ResearchReport` (exactly three risks, each with evidence), written from a compact digest of this session's tool results.
- **check_report**: deterministic checks, no LLM. Each risk must cite a tool that succeeded. The hedge's volatility numbers must match a `calculate_volatility` result. Any tool that only failed must be named in `data_gaps`. A failure sends the agent back with the specific gaps, which is a replan cycle.
- Limits: 8 tool rounds (plus 3 per revision), 2 revisions, LangGraph recursion limit 60. If both LLM providers are down, the run ends with `status="failed"` instead of an exception.

In [7]:
# AI-ASSISTED: Claude Code (claude-opus-5-5), Prompt: 'Implement the Task 3A plan (the plan approved in Entry 13)', Date: 2026-10-09 (see CITATIONS.md Entry 14)

from task3_agentic.src import prompts
from task3_agentic.src.agent import build_research_graph, default_runnables
from task3_agentic.src.tools import build_tools

preview_ctx = SessionContext(config.TICKER, trace=TraceLogger(None))
preview_tools = build_tools(preview_ctx)
models = default_runnables(preview_tools)
graph = build_research_graph(preview_ctx, models["agent"], models["report"], preview_tools)
print(graph.get_graph().draw_mermaid())
print("Tools bound to the agent:", [t.name for t in preview_tools])
print("\n--- Agent system prompt ---")
print(prompts.AGENT_SYSTEM.format(max_rounds=config.MAX_TOOL_ROUNDS, today="<run date>", ticker=config.TICKER))

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	agent(agent)
	tools(tools)
	write_report(write_report)
	check_report(check_report)
	__end__([<p>__end__</p>]):::last
	__start__ --> agent;
	agent -.-> tools;
	agent -.-> write_report;
	check_report -.-> __end__;
	check_report -.-> agent;
	tools --> agent;
	write_report --> check_report;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

Tools bound to the agent: ['get_price_data', 'get_news', 'calculate_volatility', 'llm_sentiment', 'web_search']

--- Agent system prompt ---
You are an autonomous equity research agent. Answer the user's research question with evidence you gather yourself using your tools.

Your tools:
- get_price_data(ticker, period): prices, technical indicators, momentum, 52-week range, fundamentals.
- get_news(ticker, n): recent headlines.
- calculate_volatility(ticker, window): annualised historical volatility, its

## 4. Agent Run: Observe and Replan Trace

The query from the specification, on AAPL. The trace prints live:

- **DECIDE**: the model's note for its next move. gpt-oss returns tool calls with empty text, so its reasoning summary is shown instead, marked `(reasoning)`.
- **CALL**: the tool and the arguments the model chose.
- **OBSERVE**: the tool's status and key numbers.

Every DECIDE after a tool result is numbered as a cycle and names the observation it followed. **REPLAN** marks a report check that sent the agent back.

In [8]:
# AI-ASSISTED: Claude Code (claude-opus-5-5), Prompt: 'Implement the Task 3A plan (the plan approved in Entry 13)', Date: 2026-10-09 (see CITATIONS.md Entry 14)

from task3_agentic.src.agent import run_research
from task3_agentic.src.display import TracePrinter

ctx = SessionContext(config.TICKER, trace=trace)
run = run_research(ctx, on_message=TracePrinter())
print("\nStatus:", run.final.status, "| revisions:", run.final.revisions, "| duration:", run.duration_s, "s")
print("Agent turns by model:", run.final.agent_models, "| report written by:", run.final.report_model)
print("Graph events:", run.events or "none")

----------------------------------------------------------------------------------------------------
QUERY    Analyse the current financial health and market sentiment of AAPL. Identify the top three risks to its share price over the next 90 days and suggest one data-driven hedge strategy.
----------------------------------------------------------------------------------------------------
01:49:51 | WARNING | task3.llm | Model openai/gpt-oss-120b failed: RateLimitError: HTTP 429, daily limit reached


01:49:51 | WARNING | task3.llm | openai/gpt-oss-120b skipped for the rest of this session (daily limit)


DECIDE   [step 1 | openai/gpt-oss-20b]
         (reasoning) Need price data, news, volatility, sentiment, web search.
CALL     get_price_data({"period": "1y", "ticker": "AAPL"})


OBSERVE  get_price_data -> Apple Inc.: price 340.42, 1y return 32.4%, RSI 61.23, momentum Strong Bullish, fwd P/E 35.5067, margin 27.62% [source=live]


DECIDE   [step 2 | openai/gpt-oss-20b] cycle 1: after observing get_price_data
         (reasoning) Need news, sentiment, volatility, web search.
CALL     get_news({"n": 10, "ticker": "AAPL"})


...

DECIDE   [step 5 | openai/gpt-oss-20b] cycle 4: after observing calculate_volatility
         (reasoning) Need web_search for analyst commentary.
CALL     web_search({"query": "Apple stock analyst upgrade downgrade price target 2026 October"})


OBSERVE  web_search -> 6 hits via text for 'Apple stock analyst upgrade downgrade price target 2026 October'; top: 'Apple Inc. (AAPL) Stock Forecast & Price Target — Oct 2026'


DECIDE   [step 6 | openai/gpt-oss-20b] cycle 5: after observing web_search
         **Key Findings (as of 2026‑10‑08)** | Metric | Value | Comment | |--------|-------|---------| | Current price | **$340.42** | Near 52‑week high (345.34) | | 1‑yr return | **+32.4 %** | Strong upside, but near top of range | | 52‑week % from high | **‑1.4 %** | Slightly below peak | | SMA‑50 / SMA‑200 | 322.29 / 290....
FINISH   no tool call -> write the report from the gathered evidence


01:51:55 | WARNING | task3.agent | Report writer failed: OutputParserException: Failed to parse ResearchReport from completion {"ticker": "AAPL", "financial_health": {"summary": "Apple’s share price of $340.42 sits just 1.4 % below its 52‑week high of $345.34, while both the 50‑day (322.29) and 200‑day (290.07) moving averages remain well above the curren



Status: validated | revisions: 0 | duration: 168.7 s
Agent turns by model: {'openai/gpt-oss-20b': 6} | report written by: openai/gpt-oss-20b
Graph events: none


In [9]:
# AI-ASSISTED: Claude Code (claude-opus-5-5), Prompt: 'Implement the Task 3A plan (the plan approved in Entry 13)', Date: 2026-10-09 (see CITATIONS.md Entry 14)

from collections import Counter

from task3_agentic.src.display import tool_call_table

# The order below was chosen by the model at run time; nothing in the code fixes it.
display(tool_call_table(ctx))
print("Tool order chosen by the agent:", " -> ".join(run.tool_sequence))
print("Distinct tools used:", len(set(run.tool_sequence)), "of 5 |", dict(Counter(run.tool_sequence)))

,#,tool,args,status,source,duration_ms
0,1,get_price_data,"{""ticker"": ""AAPL"", ""period"": ""1y""}",ok,live,1492
1,2,get_news,"{""ticker"": ""AAPL"", ""n"": 10}",ok,live,2319
2,3,llm_sentiment,"{""headlines"": [""Apple Rises 24% YTD: Should Yo...",ok,groq/openai/gpt-oss-20b,9172
3,4,calculate_volatility,"{""ticker"": ""AAPL"", ""window"": 20}",ok,session,13
4,5,web_search,"{""query"": ""Apple stock analyst upgrade downgra...",ok,ddgs,5469


Tool order chosen by the agent: get_price_data -> get_news -> llm_sentiment -> calculate_volatility -> web_search
Distinct tools used: 5 of 5 | {'get_price_data': 1, 'get_news': 1, 'llm_sentiment': 1, 'calculate_volatility': 1, 'web_search': 1}


In [10]:
# AI-ASSISTED: Claude Code (claude-opus-5-5), Prompt: 'Implement the Task 3A plan (the plan approved in Entry 13)', Date: 2026-10-09 (see CITATIONS.md Entry 14)

from IPython.display import Markdown

from task3_agentic.src.report import render_markdown, save_report

company = ctx.company_names.get(config.TICKER)
md_path, json_path = save_report(run.final, company)
print("Saved:", md_path.relative_to(config.REPO_ROOT), "and", json_path.name)
display(Markdown(render_markdown(run.final, company)))

Saved: task3_agentic\outputs\AAPL_2026-10-08_research_report.md and AAPL_2026-10-08_research_report.json


# Apple Inc. (AAPL) - Agent Research Report

*Generated 2026-10-08T20:22:40+00:00 | session `c2e2b8452ead` | status: **validated** | report revisions: 0 | tools used: calculate_volatility, get_news, get_price_data, llm_sentiment, web_search*

*Agent turns by model: openai/gpt-oss-20b x6 | report written by: openai/gpt-oss-20b*

> Analyse the current financial health and market sentiment of AAPL. Identify the top three risks to its share price over the next 90 days and suggest one data-driven hedge strategy.

## 1. Financial Health Summary

Apple’s share price of $340.42 sits just 1.4 % below its 52‑week high of $345.34, while both the 50‑day (322.29) and 200‑day (290.07) moving averages remain well above the current level, confirming a long‑term uptrend. Momentum is mixed: the RSI of 61.2 is in the neutral zone, but the MACD histogram of –0.43 signals short‑term bearish pressure. Valuation is tight, with an analyst consensus target of $339.35, only 0.3 % below the current price, and a forward P/E of 35.5. Profitability remains strong, with a profit margin of 27.6 % and free cash flow of $107.7 B, while the debt‑to‑equity ratio of 78.4 % indicates moderate leverage. The 90‑day expected move of ±8.33 % and an annualised volatility of 16.8 % suggest limited upside potential in the near term.

**Market sentiment.** The sentiment score of +0.12 is neutral, supported by three positive headlines such as “Apple (AAPL) Stock Trades Above Fair Value On Cash Flow Estimates” and “Apple (AAPL) Has a New Catalyst That Could Drive Its Next Upgrade Cycle.” Two negative headlines – “Nvidia Earned About Twice as Much as Apple Last Quarter. Its Stock Is Worth Only About 20% More.” and “Apple (AAPL) Stock Looks Fully Priced On Its 145% Five Year Run” – temper enthusiasm, indicating that investors are wary of valuation compression.

| Metric | Value | Source tool |
|---|---|---|
| Current price | $340.42 | `get_price_data` |
| 1‑yr return | +32.4 % | `get_price_data` |
| 52‑week % from high | ‑1.4 % | `get_price_data` |
| SMA‑50 | 322.29 | `get_price_data` |
| RSI‑14 | 61.2 | `get_price_data` |
| MACD histogram | ‑0.43 | `get_price_data` |
| 90‑day expected move | ±8.33 % | `calculate_volatility` |
| Annualised vol (20 d) | 16.8 % | `calculate_volatility` |
| Debt‑to‑equity | 78.4 % | `get_price_data` |
| Free cash flow | $107.7 B | `get_price_data` |
| Analyst consensus target | $339.35 | `web_search` |
| Sentiment score | +0.12 | `llm_sentiment` |

## 2. Top Three Risks (next 90 days)

### Risk 1: Earnings‑report surprise

A modest earnings miss could push the share price below the 1‑σ low of $313.2, as the current price is only 0.3 % above the analyst target and the MACD histogram is negative, indicating potential short‑term pullback.

Evidence:
- `get_price_data`: MACD histogram of –0.43 (negative)
- `web_search`: Analyst consensus target of $339.35, 0.3 % below current price
- `calculate_volatility`: 90‑day expected move ±8.33 % (band_1sigma_low $313.2)

### Risk 2: Supply‑chain / component shortages

Apple’s heavy reliance on global supply chains could delay product launches, potentially eroding the 16.4 % YoY revenue growth and impacting near‑term earnings.

Evidence:
- `get_price_data`: Revenue growth of 16.4 % (YoY)
- `get_news`: Headline “Apple (AAPL) Has a New Catalyst That Could Drive Its Next Upgrade Cycle” indicating upcoming product cycle

### Risk 3: Regulatory / antitrust scrutiny

Valuation concerns and potential regulatory actions could weigh on the stock, as highlighted by negative headlines and a high 5‑year run.

Evidence:
- `get_news`: Negative headline “Apple (AAPL) Stock Looks Fully Priced On Its 145% Five Year Run”
- `get_news`: Negative headline “Nvidia Earned About Twice as Much as Apple Last Quarter. Its Stock Is Worth Only About 20% More.”

## 3. Hedge Strategy Recommendation

**Strategy:** Protective put  
**Instrument:** AAPL 90‑day put, strike $313  
**Volatility basis:** annualised 16.8%, 90-day 1-sigma move 8.33%

**Rationale.** The put protects against a downside move beyond the 1‑σ low, which is likely if earnings miss or supply‑chain delays occur. It also covers the risk of regulatory pressure that could compress valuation.

**Sizing.** Buy 1 put per 100 shares (strike $313) to hedge roughly 8 % of the current price, matching the expected 90‑day move.

**Trade-offs.** Premium cost reduces upside potential beyond $313; the hedge limits gains if the stock rallies above the strike.

---

*This report was produced automatically by an LLM agent for a technical assessment. It is not investment advice or a recommendation to trade any security. Market data, headlines and search results can be delayed, incomplete or wrong, and volatility-based ranges are statistical estimates, not forecasts.*


## 5. Tool Failure and Fallback Demonstration

The same query again, with two failures injected through `FaultConfig`:

- `web_search` **always** fails with a simulated DuckDuckGo rate limit.
- `get_price_data` fails on its **first** call with a simulated Yahoo timeout.

Faults are raised inside `run_traced`, the same path a real exception takes. So the agent sees the same `error` envelope and hint it would get in production. Nothing tells the agent which tools are broken. It has to read the observation and pick an alternative.

In [11]:
# AI-ASSISTED: Claude Code (claude-opus-5-5), Prompt: 'Implement the Task 3A plan (the plan approved in Entry 13)', Date: 2026-10-09 (see CITATIONS.md Entry 14)

from task3_agentic.src.session import FaultConfig

faults = FaultConfig({"web_search": "always", "get_price_data": "first"})
fault_ctx = SessionContext(config.TICKER, trace=trace, faults=faults)
fault_run = run_research(fault_ctx, on_message=TracePrinter())
print("\nStatus:", fault_run.final.status, "| revisions:", fault_run.final.revisions, "| duration:", fault_run.duration_s, "s")
print("Agent turns by model:", fault_run.final.agent_models, "| report written by:", fault_run.final.report_model)
print("Graph events:", fault_run.events or "none")

----------------------------------------------------------------------------------------------------
QUERY    Analyse the current financial health and market sentiment of AAPL. Identify the top three risks to its share price over the next 90 days and suggest one data-driven hedge strategy.
----------------------------------------------------------------------------------------------------


01:52:40 | WARNING | task3.llm | Model openai/gpt-oss-120b failed: RateLimitError: HTTP 429, daily limit reached


01:52:40 | WARNING | task3.llm | openai/gpt-oss-120b skipped for the rest of this session (daily limit)


01:52:40 | WARNING | task3.llm | Model openai/gpt-oss-20b failed: RateLimitError: HTTP 429, daily limit reached


01:52:40 | WARNING | task3.llm | openai/gpt-oss-20b skipped for the rest of this session (daily limit)


...

01:52:43 | WARNING | task3.llm | google/gemma-4-31b-it:free skipped for the rest of this session (daily limit)


01:52:43 | ERROR   | task3.agent | Agent LLM failed on every provider (AllModelsFailed: openai/gpt-oss-120b: RateLimitError: HTTP 429, daily limit reached; openai/gpt-oss-20b: RateLimitError: HTTP 429, daily limit reached; nvidia/nemotron-3-super-120b-a12b:free: OpenAIRateLimitError: HTTP 429, daily limit reached; google/gemma-4-31b-it:free: OpenAIRateLimitError: HTTP 429, daily limit reached)


01:52:43 | WARNING | task3.agent | Report writer failed: AllModelsFailed: every model is skipped for this session (daily limits)


01:52:43 | WARNING | task3.agent | Report writer failed: AllModelsFailed: every model is skipped for this session (daily limits)



Status: failed | revisions: 0 | duration: 3.7 s
Agent turns by model: {} | report written by: None
Graph events: ['Agent LLM failed on every provider (AllModelsFailed: openai/gpt-oss-120b: RateLimitError: HTTP 429, daily limit reached; openai/gpt-oss-20b: RateLimitError: HTTP 429, daily limit reached; nvidia/nemotron-3-super-120b-a12b:free: OpenAIRateLimitError: HTTP 429, daily limit reached; google/gemma-4-31b-it:free: OpenAIRateLimitError: HTTP 429, daily limit reached)']


In [12]:
# AI-ASSISTED: Claude Code (claude-opus-5-5), Prompt: 'Implement the Task 3A plan (the plan approved in Entry 13)', Date: 2026-10-09 (see CITATIONS.md Entry 14)

from task3_agentic.src.display import trace_table

display(tool_call_table(fault_ctx))
# The same calls as they were written to agent_trace.jsonl (output truncated to 200 characters).
fault_trace = trace_table(TraceLogger.read(config.TRACE_FILE), session_id=fault_ctx.session_id)
failed_calls = fault_trace[fault_trace["status"] != "ok"] if not fault_trace.empty else fault_trace
if failed_calls.empty:
    print("No failed tool calls were logged for this session.")
else:
    with pd.option_context("display.max_colwidth", 90):
        display(failed_calls[["tool", "args", "status", "duration_ms", "output"]])

""


No failed tool calls were logged for this session.


In [13]:
# AI-ASSISTED: Claude Code (claude-opus-5-5), Prompt: 'Implement the Task 3A plan (the plan approved in Entry 13)', Date: 2026-10-09 (see CITATIONS.md Entry 14)

company = fault_ctx.company_names.get(config.TICKER)
fault_md, _ = save_report(fault_run.final, company, suffix="_fault_demo")
print("Saved:", fault_md.relative_to(config.REPO_ROOT))
display(Markdown(render_markdown(fault_run.final, company)))

Saved: task3_agentic\outputs\AAPL_2026-10-08_research_report_fault_demo.md


# AAPL - Agent Research Report

*Generated 2026-10-08T20:22:43+00:00 | session `0b580b84c86d` | status: **failed** | report revisions: 0 | tools used: none*

*Agent turns by model: n/a | report written by: n/a*

> Analyse the current financial health and market sentiment of AAPL. Identify the top three risks to its share price over the next 90 days and suggest one data-driven hedge strategy.

## Report unavailable

AllModelsFailed: every model is skipped for this session (daily limits)

## Validation warnings

- The report could not be produced (AllModelsFailed: every model is skipped for this session (daily limits)).

---

*This report was produced automatically by an LLM agent for a technical assessment. It is not investment advice or a recommendation to trade any security. Market data, headlines and search results can be delayed, incomplete or wrong, and volatility-based ranges are statistical estimates, not forecasts.*


### 3A evidence summary

The table below is computed from the two runs above, not typed by hand.

In [14]:
# AI-ASSISTED: Claude Code (claude-opus-5-5), Prompt: 'Implement the Task 3A plan (the plan approved in Entry 13)', Date: 2026-10-09 (see CITATIONS.md Entry 14)

from langchain_core.messages import AIMessage, ToolMessage

from task3_agentic.src.agent import count_tool_rounds


def cycles(messages):
    """Decisions taken right after observing at least one tool result."""
    return sum(1 for prev, cur in zip(messages, messages[1:]) if isinstance(prev, ToolMessage) and isinstance(cur, AIMessage))


def errored_tools(session):
    """Tools that returned error or empty at least once in a session."""
    return sorted({r.tool for r in session.history if r.result.status != "ok"})


report = run.final.report
summary_rows = [
    ("Tools implemented and called", f"{len(set(run.tool_sequence))} distinct of 5 in the main run; all 5 called directly in section 2"),
    ("Order chosen by the model", " -> ".join(run.tool_sequence)),
    ("Order under faults", " -> ".join(fault_run.tool_sequence)),
    ("Observe -> decide cycles", f"main run {cycles(run.messages)}, fault run {cycles(fault_run.messages)}"),
    ("Report-check replans", f"main run {run.final.revisions}, fault run {fault_run.final.revisions}"),
    ("Report status", f"main {run.final.status}, fault {fault_run.final.status}"),
    ("Report sections", "Financial Health Summary / Top Three Risks / Hedge Strategy" if report else "missing"),
    ("Evidence items per risk", [len(r.evidence) for r in report.risks] if report else "-"),
    ("Tools that errored in the fault run", errored_tools(fault_ctx)),
    ("Tools that never recovered (recorded in data_gaps)", sorted(fault_ctx.failed_tools())),
    ("Models that answered", f"main {run.final.agent_models} / report {run.final.report_model}; fault {fault_run.final.agent_models} / report {fault_run.final.report_model}"),
    ("Unhandled exceptions", "none (both runs returned a FinalReport)"),
    ("Trace lines written", len(TraceLogger.read(config.TRACE_FILE))),
]
with pd.option_context("display.max_colwidth", 200):
    display(pd.DataFrame(summary_rows, columns=["Check", "Result"]))

,Check,Result
0,Tools implemented and called,5 distinct of 5 in the main run; all 5 called directly in section 2
1,Order chosen by the model,get_price_data -> get_news -> llm_sentiment -> calculate_volatility -> web_search
2,Order under faults,
3,Observe -> decide cycles,"main run 5, fault run 0"
4,Report-check replans,"main run 0, fault run 0"
5,Report status,"main validated, fault failed"
6,Report sections,Financial Health Summary / Top Three Risks / Hedge Strategy
7,Evidence items per risk,"[3, 2, 2]"
8,Tools that errored in the fault run,[]
9,Tools that never recovered (recorded in data_gaps),[]


# Part 3B - Multi-Agent Coordination

Two agents with separate roles, separate conversations and separate tool access:

| Agent | Role | Tools (enforced) | Output |
|---|---|---|---|
| A, `data_analyst` | Quantitative analysis | `get_price_data`, `calculate_volatility`, `llm_sentiment` | `DataBrief` (structured JSON) |
| B, `research_writer` | Qualitative synthesis | `web_search`, `get_news` | Final `ResearchReport` |

```
orchestrator --task--> A --DataBrief--> B --ClarificationRequest--> A --ClarificationResponse--> B --ResearchReport--> done
```

**How Agent A gets headlines.** A has `llm_sentiment` but no news access, so its first brief carries no sentiment. B fetches the headlines. B's one clarification request carries them back to A, A scores them, and B uses the score. Neither agent can produce the sentiment evidence alone.

## 6. Handoff Schemas

Every inter-agent message is a Pydantic model inside an `AgentMessage` envelope (sender, recipient, kind, payload, timestamp). Nothing is passed as a raw string. The receiver re-validates the payload against the schema it expects (`AgentMessage.open`).

The **numbers** in `DataBrief` and `ClarificationResponse` are filled in code from Agent A's own tool results. A's LLM writes only the interpretive fields (`key_findings`, `quant_risk_flags`, `confidence_notes`, `answer`).

In [ ]:
# AI-ASSISTED: Claude Code (claude-opus-5-5), Prompt: 'Implement the Task 3B plan', Date: 2026-10-09 (see CITATIONS.md Entry 16)

from task3_agentic.src.display import schema_table
from task3_agentic.src.handoff import AgentMessage, ClarificationRequest, ClarificationResponse, DataBrief

for schema in (DataBrief, ClarificationRequest, ClarificationResponse, AgentMessage):
    print(f"\n{schema.__name__}")
    with pd.option_context("display.max_colwidth", 110):
        display(schema_table(schema))

**Tool restriction is enforced in code, three ways:**

1. **Binding.** Each agent's LLM is bound only to its own tool schemas, so it never sees the others.
2. **ToolNode.** Each agent's loop can only execute its own tools. A call to any other name, which a confused or manipulated model might forge, comes back as an error message, and the tool never runs.
3. **Toolkit guard.** Each agent's `ResearchTools` has an `allowed` set, checked in `run_traced`. Even code that calls the toolkit directly is refused, and the attempt is written to `agent_trace.jsonl`.

The cell below exercises all three without any LLM call. The forged call in layer 2 comes from a two-line scripted model.

A related guard: `llm_sentiment` only scores headlines that `get_news` or `web_search` actually returned in this session. This was added after a live run in which Agent A, which has no news access, invented headlines and scored them.

In [ ]:
# AI-ASSISTED: Claude Code (claude-opus-5-5), Prompt: 'Implement the Task 3B plan', Date: 2026-10-09 (see CITATIONS.md Entry 16)

from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.runnables import RunnableLambda

from task3_agentic.src.multi_agent import build_agent_toolkits
from task3_agentic.src.react import build_react_loop, run_react_loop

A_NAME, B_NAME = config.AGENT_A_NAME, config.AGENT_B_NAME
demo_ctx = SessionContext(config.TICKER, trace=trace)
demo_toolkits = build_agent_toolkits(demo_ctx)

# Layer 1: the tool schemas each agent's LLM is bound to.
display(pd.DataFrame(
    [{"agent": agent, "bound tools": ", ".join(t.name for t in tools)} for agent, (_, tools) in demo_toolkits.items()]
))

# Layer 2: a model forges a call to a tool its agent does not have; the ToolNode refuses it.
forged = iter([
    AIMessage(content="", tool_calls=[{"name": "web_search", "args": {"query": "Apple"}, "id": "forged_1", "type": "tool_call"}]),
    AIMessage(content="done"),
])
loop_a = build_react_loop(RunnableLambda(lambda _: next(forged)), demo_toolkits[A_NAME][1], "Agent A", max_rounds=2)
rejected = next(m for m in run_react_loop(loop_a, [HumanMessage(content="go")])["messages"] if isinstance(m, ToolMessage))
print(f"Layer 2  data_analyst -> web_search via ToolNode: status={rejected.status}: {rejected.content[:120]}")

# Layer 3: direct calls past the LLM are refused by the toolkit guard (and logged).
toolkit_a, toolkit_b = demo_toolkits[A_NAME][0], demo_toolkits[B_NAME][0]
for agent, result in (
    (A_NAME, toolkit_a.web_search("Apple analyst outlook")),
    (A_NAME, toolkit_a.get_news(config.TICKER, 5)),
    (B_NAME, toolkit_b.get_price_data(config.TICKER, "1y")),
):
    print(f"Layer 3  {agent} -> {result.tool}: status={result.status}: {result.error}")

# Headline guard: A cannot score a headline no news tool returned.
invented = toolkit_a.llm_sentiment(["Apple reports Q3 earnings beat expectations"])
print(f"Guard    data_analyst -> llm_sentiment(invented headline): status={invented.status}: {invented.error}")

denied = trace_table(TraceLogger.read(config.TRACE_FILE), session_id=demo_ctx.session_id)
with pd.option_context("display.max_colwidth", 80):
    display(denied[["agent", "tool", "status", "output"]])

## 7. Agent A (Data Analyst) and Agent B (Research Writer)

Each agent is a ReAct loop (`src/react.py`, shared with 3A): the LLM picks its own tool calls from its own tool list. The parent graph below runs the four steps in order. Within each step the agent decides what to call.

| Step | Agent | Tool rounds | Structured output |
|---|---|---|---|
| `analyst_initial` | A | up to 5 | `AnalystInterpretation`, merged into the code-filled `DataBrief` |
| `writer_research` | B | up to 5 | exactly one `ClarificationRequest` (B's LLM picks kind and question) |
| `analyst_clarify` | A, same conversation | up to 3 more | `ClarificationResponse` (values from A's new tool results) |
| `writer_final` | B | none | `ResearchReport`, checked by `check_report` + `check_incorporation`, one repair pass |

In [ ]:
# AI-ASSISTED: Claude Code (claude-opus-5-5), Prompt: 'Implement the Task 3B plan', Date: 2026-10-09 (see CITATIONS.md Entry 16)

from task3_agentic.src.multi_agent import build_pipeline_graph, default_pipeline_models

preview_ctx = SessionContext(config.TICKER, trace=TraceLogger(None))
preview = build_agent_toolkits(preview_ctx)
pipeline = build_pipeline_graph(
    preview_ctx,
    default_pipeline_models(preview[A_NAME][1], preview[B_NAME][1]),
    preview[A_NAME][0],
    preview[A_NAME][1],
    preview[B_NAME][1],
    config.RESEARCH_QUERY_TEMPLATE.format(ticker=config.TICKER),
    None,
)
print(pipeline.get_graph().draw_mermaid())
for title, text in (("Agent A system prompt", prompts.ANALYST_SYSTEM), ("Agent B system prompt", prompts.WRITER_SYSTEM)):
    print(f"--- {title} ---")
    print(text.format(today="<run date>", ticker=config.TICKER))

## 8. End-to-End Pipeline Run with Full Message Trace

One call runs the whole pipeline: query in, report out, with no manual step. The trace prints live:

- `[A data_analyst]` and `[B research_writer]` prefix each agent's DECIDE / CALL / OBSERVE lines.
- A `HANDOFF` block shows every inter-agent message: sender, recipient, kind, and the validated payload.
- `[pipeline]` lines are graph events (budget reached, model fallbacks, repair passes).

In [ ]:
# AI-ASSISTED: Claude Code (claude-opus-5-5), Prompt: 'Implement the Task 3B plan', Date: 2026-10-09 (see CITATIONS.md Entry 16)

from task3_agentic.src.display import MultiAgentPrinter
from task3_agentic.src.multi_agent import run_multi_agent

from task3_agentic.src.cache import multi_agent_with_cache

ma_ctx = SessionContext(config.TICKER, trace=trace)
# force_refresh: this run is always live (a same-day re-execution must not load yesterday's work from the
# cache); it also writes the cache that section 11 reads. AI-ASSISTED: Claude Code (claude-opus-5-5), Prompt: 'Implement the Task 3C plan', Date: 2026-10-09 (see CITATIONS.md Entry 18)
first_cache = multi_agent_with_cache(ma_ctx, force_refresh=True, events=events, observer=MultiAgentPrinter())
ma_run = first_cache.run
print("\nCache:", first_cache.note)
print("\nStatus:", ma_run.final.status, "| report repair passes:", ma_run.final.revisions, "| duration:", ma_run.duration_s, "s")
print("Agent turns by model:", ma_run.final.agent_models, "| report written by:", ma_run.final.report_model)
print("Graph events:", ma_run.events or "none")

In [ ]:
# AI-ASSISTED: Claude Code (claude-opus-5-5), Prompt: 'Implement the Task 3B plan', Date: 2026-10-09 (see CITATIONS.md Entry 16)

from task3_agentic.src.display import handoff_table

print("Inter-agent messages (in order):")
with pd.option_context("display.max_colwidth", 120):
    display(handoff_table(ma_run.handoffs))
print("Tool calls, with the agent that made each one:")
display(tool_call_table(ma_ctx))
lines = trace_table(TraceLogger.read(config.TRACE_FILE), session_id=ma_ctx.session_id)
print("agent_trace.jsonl lines for this run, per agent:", lines.groupby("agent")["tool"].apply(list).to_dict())

In [ ]:
# AI-ASSISTED: Claude Code (claude-opus-5-5), Prompt: 'Implement the Task 3B plan', Date: 2026-10-09 (see CITATIONS.md Entry 16)

from task3_agentic.src.multi_agent import save_handoffs

company = ma_ctx.company_names.get(config.TICKER)
ma_md, _ = save_report(ma_run.final, company, suffix=config.MULTI_AGENT_REPORT_SUFFIX)
handoffs_path = save_handoffs(ma_run)
print("Saved:", ma_md.relative_to(config.REPO_ROOT), "and", handoffs_path.relative_to(config.REPO_ROOT))
display(Markdown(render_markdown(ma_run.final, company)))

## 9. Critique Loop: Request, Response, Incorporation

The critique loop from the run above: B's one request, A's answer, and where B's final report uses it. `check_incorporation` reads every number in the report and confirms that A's returned value is among them. It also requires `clarification_used` to say what was asked and how the answer was used.

In [ ]:
# AI-ASSISTED: Claude Code (claude-opus-5-5), Prompt: 'Implement the Task 3B plan', Date: 2026-10-09 (see CITATIONS.md Entry 16)

from task3_agentic.src.handoff import check_incorporation, expected_value

request_message = next(h for h in ma_run.handoffs if h.kind == "clarification_request")
response_message = next(h for h in ma_run.handoffs if h.kind == "clarification_response")
print(f"REQUEST  {request_message.sender} -> {request_message.recipient}{' (fallback)' if request_message.fallback else ''}")
print(json.dumps(request_message.payload, indent=2, ensure_ascii=False))
answer = dict(response_message.payload)
for key in ("sentiment", "volatility"):
    if key in answer:  # keep the printout short: the full block is in the handoffs file
        answer[key] = {k: answer[key][k] for k in list(answer[key])[:8]}
print(f"\nRESPONSE {response_message.sender} -> {response_message.recipient}")
print(json.dumps(answer, indent=2, ensure_ascii=False))

if ma_run.final.report is not None:
    target = expected_value(ma_run.response)
    print("\nValue B had to use:", target[2] + " = " + str(target[0]) if target else "none (A's answer failed)")
    print("B's clarification_used:", ma_run.final.report.clarification_used)
    print("check_incorporation issues:", check_incorporation(ma_run.final.report, ma_run.response) or "none")

### 3B evidence summary

Computed from the run above and the section 6 enforcement demo.

In [ ]:
# AI-ASSISTED: Claude Code (claude-opus-5-5), Prompt: 'Implement the Task 3B plan', Date: 2026-10-09 (see CITATIONS.md Entry 16)

from task3_agentic.src.handoff import AnalystTask
from task3_agentic.src.schemas import ResearchReport

expected_schema = {
    "task": AnalystTask,
    "data_brief": DataBrief,
    "clarification_request": ClarificationRequest,
    "clarification_response": ClarificationResponse,
    "final_report": ResearchReport,
}
revalidated = all(expected_schema[h.kind].model_validate(h.payload) is not None for h in ma_run.handoffs)
crossed = [r.tool for r in ma_ctx.history if r.tool not in (config.AGENT_A_TOOLS if r.agent == A_NAME else config.AGENT_B_TOOLS)]
demo_denied = [r for r in demo_ctx.history if r.denied]
rows = [
    ("Bound tools", ma_run.bound_tools),
    ("Tool calls outside an agent's list in the run", crossed or "none"),
    ("Calls refused by the guard in the section 6 demo", len(demo_denied)),
    ("Handoff kinds, in order", " -> ".join(h.kind for h in ma_run.handoffs)),
    ("Every payload re-validates against its schema", revalidated),
    ("Critique rounds", ma_run.critique_rounds),
    ("Request", f"{ma_run.request.request_kind}: {ma_run.request.question}" if ma_run.request else "-"),
    ("Answer", f"{ma_run.response.status}, fulfilled by {ma_run.response.fulfilled_by}" if ma_run.response else "-"),
    ("Incorporation issues", check_incorporation(ma_run.final.report, ma_run.response) if ma_run.final.report else "no report"),
    ("Tool calls per agent", {a: ma_run.tool_sequence(a) for a in (A_NAME, B_NAME)}),
    ("Final status", ma_run.final.status),
    ("Manual steps between query and report", 0),
]
with pd.option_context("display.max_colwidth", 200):
    display(pd.DataFrame(rows, columns=["Check", "Result"]))

# Part 3C - Memory and Observability

| Requirement | Mechanism | Shown in |
|---|---|---|
| Short-term memory | The 3A research graph is compiled with a LangGraph checkpointer (`InMemorySaver`). The follow-up continues the same thread, so the agent's earlier tool results are in its context. | §10 |
| Persistent cache | `cache/<TICKER>_<YYYY-MM-DD>_<pipeline>.json`, date in US/Eastern. A same-day call loads it with no tool calls. | §8 (write), §11 (hit) |
| `agent_trace.jsonl` | Every tool call: timestamp, session, agent, tool, args, status, output (200 chars), duration. Cache and memory events go to `logs/session_events.jsonl`. | §12 |

## 10. Short-Term Memory: Follow-Up Question Without Re-Calling Tools

The follow-up is a new message on the **section 4 run's own thread**. The agent has its five tools bound and may call one (it gets one tool round, then must answer). It is told that its earlier tool results are its memory. In follow-up mode, an answer without a tool call ends the turn, so no new report is written.

The evidence below is computed, not asserted:
- tool calls during the follow-up;
- `agent_trace.jsonl` line count before and after;
- whether the answer cites the values of the earlier `calculate_volatility` result.

In [ ]:
# AI-ASSISTED: Claude Code (claude-opus-5-5), Prompt: 'Implement the Task 3C plan', Date: 2026-10-09 (see CITATIONS.md Entry 18)

from task3_agentic.src.display import followup_table
from task3_agentic.src.memory import ask_followup, volatility_memory

question = "Which volatility window did you compute, and what 90-day 1-sigma price band did it give?"
thread = {"configurable": {"thread_id": run.thread_id}}
in_thread = len(run.graph.get_state(thread).values.get("messages", [])) if run.graph is not None else 0
print(f"Thread {run.thread_id}: {in_thread} messages already in memory (the section 4 run)")
remembered = volatility_memory(run)
print("Earlier calculate_volatility result:", remembered or "none (section 4 did not compute volatility)")

followup = ask_followup(run, question, remembered=remembered, on_message=TracePrinter(), events=events)
print("\nAnswer:", followup.answer or followup.error)
with pd.option_context("display.max_colwidth", 140):
    display(followup_table(followup))

## 11. Persistent Cache: Second Run Loads the Cached Brief

Section 8 ran the two-agent pipeline live (`force_refresh=True`) and wrote today's brief to the cache. The cell below asks for the same ticker again **as a new session**. `multi_agent_with_cache` finds `AAPL_<today>_multi_agent.json`, validates it, and returns it without running a single tool. A cached file from another date is never reused.

In [ ]:
# AI-ASSISTED: Claude Code (claude-opus-5-5), Prompt: 'Implement the Task 3C plan', Date: 2026-10-09 (see CITATIONS.md Entry 18)

import datetime as dt

from task3_agentic.src.cache import cache_path, load_cached, market_date

written = cache_path(config.TICKER, config.PIPELINE_MULTI_AGENT)
print("Market date (US/Eastern):", market_date())
print("Section 8 cache result:", first_cache.note)
if written.exists():
    stored = json.loads(written.read_text(encoding="utf-8"))
    print(f"{written.relative_to(config.REPO_ROOT)}: {written.stat().st_size:,} bytes")
    display(pd.DataFrame(
        [(key, stored[key]) for key in ("schema_version", "ticker", "cache_date", "pipeline", "created_at", "session_id", "status", "tool_calls")],
        columns=["field", "value"],
    ))

In [ ]:
# AI-ASSISTED: Claude Code (claude-opus-5-5), Prompt: 'Implement the Task 3C plan', Date: 2026-10-09 (see CITATIONS.md Entry 18)

from task3_agentic.src.cache import multi_agent_with_cache
from task3_agentic.src.display import cache_table

if first_cache.brief is None:
    print("Section 8 produced no report, so nothing was cached; skipping the second run rather than spending quota.")
else:
    second_ctx = SessionContext(config.TICKER, trace=trace)
    lines_before = trace_line_count(config.TRACE_FILE)
    second = multi_agent_with_cache(second_ctx, events=events, observer=MultiAgentPrinter())
    lines_after = trace_line_count(config.TRACE_FILE)
    display(cache_table(second))
    print(f"agent_trace.jsonl lines: {lines_before} -> {lines_after} (no tool ran)" if lines_after == lines_before
          else f"agent_trace.jsonl lines: {lines_before} -> {lines_after}")
    display(Markdown(render_markdown(second.brief.final, ma_ctx.company_names.get(config.TICKER))))

In [ ]:
# AI-ASSISTED: Claude Code (claude-opus-5-5), Prompt: 'Implement the Task 3C plan', Date: 2026-10-09 (see CITATIONS.md Entry 18)

# A different date is a different key: yesterday's file (if any) is never served for today.
yesterday = (pd.Timestamp(market_date()) - pd.Timedelta(days=1)).strftime("%Y-%m-%d")
print(f"load_cached({config.TICKER!r}, multi_agent, date={yesterday!r}):", load_cached(config.TICKER, config.PIPELINE_MULTI_AGENT, date=yesterday))
print(f"load_cached({config.TICKER!r}, single_agent, today):", load_cached(config.TICKER, config.PIPELINE_SINGLE_AGENT))

## 12. Trace Log Inspection (`agent_trace.jsonl`)

`agent_trace.jsonl` is written by `run_traced`, the one function every tool call passes through, including calls the tool-access guard refused. `audit_trace` checks every line against the specification: required fields, output of 200 characters or less, and a duration of 0 or more.

In [ ]:
# AI-ASSISTED: Claude Code (claude-opus-5-5), Prompt: 'Implement the Task 3C plan', Date: 2026-10-09 (see CITATIONS.md Entry 18)

from task3_agentic.src.dashboard_data import load_events, load_trace, per_session
from task3_agentic.src.tracing import audit_trace

records = TraceLogger.read(config.TRACE_FILE)
audit = audit_trace(records)
display(pd.DataFrame(
    [
        ("Lines (tool calls)", audit["lines"]),
        ("Lines missing a required field", len(audit["missing_fields"])),
        (f"Outputs over {config.TRACE_OUTPUT_MAX_CHARS} characters", len(audit["output_over_limit"])),
        ("Missing or negative durations", len(audit["bad_duration"])),
        ("Calls by agent", audit["by_agent"]),
        ("Calls by tool", audit["by_tool"]),
        ("Calls by status", audit["by_status"]),
    ],
    columns=["Check", "Result"],
))
print("Sessions in this notebook run:")
display(per_session(load_trace(config.TRACE_FILE)))
print("Three raw lines:")
for record in records[:3]:
    print(json.dumps(record, ensure_ascii=False))
print("\nsession_events.jsonl:")
with pd.option_context("display.max_colwidth", 120):
    display(load_events(config.EVENTS_FILE))

# Bonus - Observability Platform

## 13. Streamlit Trace Dashboard

A Streamlit dashboard reads `agent_trace.jsonl` and `session_events.jsonl` and shows the trace visually:
- filters by session, agent, tool and status;
- KPIs (calls, errors, empty, denied, p50/p95 duration);
- a per-session table;
- a Gantt-style timeline of each session's tool calls;
- every call's arguments and logged output;
- the cache and memory events.

```
streamlit run task3_agentic/dashboard/app.py
```

The data functions behind it (`src/dashboard_data.py`) are tested offline. The cell below shows the dashboard's KPI row and session table for this notebook run.

In [ ]:
# AI-ASSISTED: Claude Code (claude-opus-5-5), Prompt: 'Implement the Task 3C plan', Date: 2026-10-09 (see CITATIONS.md Entry 18)

from task3_agentic.src.dashboard_data import summarise

trace_frame = load_trace(config.TRACE_FILE)
display(pd.DataFrame([summarise(trace_frame)]))

## Citations

See [`CITATIONS.md`](../CITATIONS.md) in the repository root. AI-assisted cells are also cited inline.